# Analyse: Hautglättung


In [ ]:
import sys
!{sys.executable} -m pip install numpy pandas opencv-python pillow tqdm torch facenet-pytorch


In [7]:
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
from PIL import Image
from tqdm import tqdm

import torch
from facenet_pytorch import MTCNN

print(torch.__version__)
print('CUDA available:', torch.cuda.is_available())


2.7.1+cpu
CUDA available: False


In [8]:
DATA_DIR = Path('../../data')

COMBINED_CSV = DATA_DIR / '03_datasets/influencer_balanced/01_AI_AND_REAL_TIKTOK_VIDEOS_stratified_per_influencer_50.csv'
OUTPUT_CSV = DATA_DIR / '04_analysis_results' / 'visual_features' / '07_AI_AND_REAL_TIKTOK_VIDEOS_stratified_with_skin_smoothness.csv'
FRAME_ROOT = DATA_DIR / '02_media/stratified_sample/frames'
SOURCE_FILTER = None
DEFAULT_MAX_FRAMES_PER_VIDEO = 60

CANDIDATE_SIGMAS = [1.5, 2.0, 2.5, 3.0, 3.5, 4.0]
CALIBRATION_VIDEOS = 40
CALIBRATION_FRAMES_PER_VIDEO = 5
DOG_RATIO = 1.6

if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f'Running on device: {device}')

df = pd.read_csv(COMBINED_CSV)
if 'influencer_type' not in df.columns and 'source' in df.columns:
    df['influencer_type'] = df['source']


Running on device: cpu


In [9]:
if SOURCE_FILTER:
    df = df[df['influencer_type'].isin(SOURCE_FILTER)].copy()

video_id_col = 'video_id' if 'video_id' in df.columns else 'id'


def get_video_id(row) -> str:
    value = row.get(video_id_col, None)
    if pd.isna(value):
        return ''
    return str(value)


def get_duration_seconds(row):
    for col in ('video_duration_seconds', 'duration_seconds', 'duration', 'video_duration'):
        if col in row.index:
            value = row.get(col, np.nan)
            if pd.notna(value):
                return value
    return np.nan


def has_frames(video_id: str) -> bool:
    return (FRAME_ROOT / video_id).is_dir()


video_ids = df[video_id_col].astype(str)
df['has_frames'] = [has_frames(video_id) for video_id in video_ids]

missing_ids = video_ids[~df['has_frames']].tolist()
print(f'Total videos in CSV: {len(df)}')
print(f'Videos with frame folder: {df["has_frames"].sum()}')
print(f'Videos missing frame folder: {len(missing_ids)}')
if missing_ids:
    print('First missing video_ids:', missing_ids[:20])

df = df[df['has_frames']].reset_index(drop=True)
print(f'Restricted to {len(df)} rows with available frames')


Total videos in CSV: 500
Videos with frame folder: 500
Videos missing frame folder: 0
Restricted to 500 rows with available frames


In [10]:
mtcnn = MTCNN(keep_all=True, device=device)


def sample_frame_paths(video_id: str, duration_seconds=None, default_max_frames: int = DEFAULT_MAX_FRAMES_PER_VIDEO):
    folder = FRAME_ROOT / video_id
    if not folder.is_dir():
        return []

    frame_files = sorted(folder.glob('*.jpeg'))
    if not frame_files:
        frame_files = sorted(folder.glob('*.jpg'))
    if not frame_files:
        frame_files = sorted(folder.glob('*.png'))
    if not frame_files:
        return []

    try:
        duration_value = float(duration_seconds)
    except (TypeError, ValueError):
        duration_value = np.nan

    if pd.notna(duration_value) and duration_value > 0:
        target_frames = int(np.ceil(duration_value))
    else:
        target_frames = default_max_frames

    target_frames = max(1, min(target_frames, len(frame_files)))
    step = max(1, len(frame_files) // target_frames)
    return frame_files[::step][:target_frames]


def detect_largest_face_roi(image_rgb: np.ndarray):
    pil_img = Image.fromarray(image_rgb)
    boxes, _ = mtcnn.detect(pil_img)
    if boxes is None or len(boxes) == 0:
        return None

    areas = [(max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])) for b in boxes]
    idx = int(np.argmax(areas))

    h, w = image_rgb.shape[:2]
    x1, y1, x2, y2 = boxes[idx]
    x1 = max(0, int(np.floor(x1)))
    y1 = max(0, int(np.floor(y1)))
    x2 = min(w, int(np.ceil(x2)))
    y2 = min(h, int(np.ceil(y2)))
    if x2 <= x1 or y2 <= y1:
        return None

    roi = image_rgb[y1:y2, x1:x2]
    return roi if roi.size > 0 else None


def highpass_var(gray: np.ndarray, sigma: float):
    blur = cv2.GaussianBlur(gray, (0, 0), sigmaX=sigma)
    high = gray.astype(np.float32) - blur.astype(np.float32)
    return float(np.var(high))


def dog_var(gray: np.ndarray, sigma: float, ratio: float = DOG_RATIO):
    blur_small = cv2.GaussianBlur(gray, (0, 0), sigmaX=sigma)
    blur_large = cv2.GaussianBlur(gray, (0, 0), sigmaX=sigma * ratio)
    dog = blur_small.astype(np.float32) - blur_large.astype(np.float32)
    return float(np.var(dog))


def calibrate_sigma(df_in: pd.DataFrame):
    per_sigma_video_means = {s: [] for s in CANDIDATE_SIGMAS}
    per_sigma_within_vars = {s: [] for s in CANDIDATE_SIGMAS}

    sample_n = min(len(df_in), CALIBRATION_VIDEOS)
    sample_df = df_in.sample(n=sample_n, random_state=42) if sample_n > 0 else df_in

    for _, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc='Calibrating sigma'):
        vid = get_video_id(row)
        frames = sample_frame_paths(vid, duration_seconds=np.nan, default_max_frames=CALIBRATION_FRAMES_PER_VIDEO)
        if not frames:
            continue

        vals = {s: [] for s in CANDIDATE_SIGMAS}
        for fp in frames:
            bgr = cv2.imread(str(fp))
            if bgr is None:
                continue
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            face = detect_largest_face_roi(rgb)
            if face is None:
                continue

            gray = cv2.cvtColor(face, cv2.COLOR_RGB2GRAY)
            if gray.shape[0] < 10 or gray.shape[1] < 10:
                continue

            for s in CANDIDATE_SIGMAS:
                vals[s].append(highpass_var(gray, s))

        for s in CANDIDATE_SIGMAS:
            if vals[s]:
                per_sigma_video_means[s].append(float(np.mean(vals[s])))
                per_sigma_within_vars[s].append(float(np.var(vals[s])))

    scores = {}
    for s in CANDIDATE_SIGMAS:
        if len(per_sigma_video_means[s]) < 3:
            scores[s] = -np.inf
            continue
        between = float(np.var(per_sigma_video_means[s]))
        within = float(np.mean(per_sigma_within_vars[s])) + 1e-8
        scores[s] = between / within

    best_sigma = max(scores, key=scores.get)
    return best_sigma, scores


SELECTED_SIGMA, SIGMA_SCORES = calibrate_sigma(df)
print('Sigma scores:', {k: round(v, 4) for k, v in SIGMA_SCORES.items()})
print(f'Selected sigma (data-driven): {SELECTED_SIGMA}')


def compute_skin_texture_metrics(face_rgb: np.ndarray, sigma: float = SELECTED_SIGMA):
    gray = cv2.cvtColor(face_rgb, cv2.COLOR_RGB2GRAY)
    if gray.shape[0] < 10 or gray.shape[1] < 10:
        return np.nan, np.nan, np.nan, np.nan, np.nan

    lap_var = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    high_var = highpass_var(gray, sigma)
    dog_energy = dog_var(gray, sigma, DOG_RATIO)

    smooth_high = float(1.0 / (1.0 + high_var))
    smooth_dog = float(1.0 / (1.0 + dog_energy))

    return lap_var, high_var, dog_energy, smooth_high, smooth_dog


Calibrating sigma: 100%|██████████| 40/40 [00:16<00:00,  2.46it/s]

Sigma scores: {1.5: 7.0523, 2.0: 7.0524, 2.5: 6.9566, 3.0: 6.9344, 3.5: 6.843, 4.0: 6.7112}
Selected sigma (data-driven): 2.0


In [11]:
video_lap_var = []
video_highpass_var = []
video_dog_var = []
video_smoothness_highpass = []
video_smoothness_dog = []
video_detected_face_frames = []

for _, row in tqdm(df.iterrows(), total=len(df), desc='Processing videos'):
    vid = get_video_id(row)
    duration_seconds = get_duration_seconds(row)
    frame_paths = sample_frame_paths(vid, duration_seconds=duration_seconds)

    lap_vals, high_vals, dog_vals = [], [], []
    smooth_high_vals, smooth_dog_vals = [], []

    for fp in frame_paths:
        bgr = cv2.imread(str(fp))
        if bgr is None:
            continue
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        face_roi = detect_largest_face_roi(rgb)
        if face_roi is None:
            continue

        lap, high, dog_e, s_high, s_dog = compute_skin_texture_metrics(face_roi, sigma=SELECTED_SIGMA)
        if not np.isnan(high):
            lap_vals.append(lap)
            high_vals.append(high)
            dog_vals.append(dog_e)
            smooth_high_vals.append(s_high)
            smooth_dog_vals.append(s_dog)

    if high_vals:
        video_lap_var.append(float(np.mean(lap_vals)))
        video_highpass_var.append(float(np.mean(high_vals)))
        video_dog_var.append(float(np.mean(dog_vals)))
        video_smoothness_highpass.append(float(np.mean(smooth_high_vals)))
        video_smoothness_dog.append(float(np.mean(smooth_dog_vals)))
        video_detected_face_frames.append(len(high_vals))
    else:
        video_lap_var.append(np.nan)
        video_highpass_var.append(np.nan)
        video_dog_var.append(np.nan)
        video_smoothness_highpass.append(np.nan)
        video_smoothness_dog.append(np.nan)
        video_detected_face_frames.append(0)


Processing videos: 100%|██████████| 500/500 [21:16<00:00,  2.55s/it]  


In [12]:
df['skin_selected_sigma'] = float(SELECTED_SIGMA)
df['skin_texture_laplacian_var'] = video_lap_var
df['skin_texture_highpass_var'] = video_highpass_var
df['skin_texture_dog_var'] = video_dog_var
df['skin_smoothness_highpass_index'] = video_smoothness_highpass
df['skin_smoothness_dog_index'] = video_smoothness_dog
df['detected_skin_face_frames'] = video_detected_face_frames

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_CSV, index=False)

print(f'Saved: {OUTPUT_CSV}')
df[['skin_selected_sigma', 'skin_texture_highpass_var', 'skin_texture_dog_var', 'skin_smoothness_highpass_index', 'skin_smoothness_dog_index', 'detected_skin_face_frames']].head(20)


Saved: ..\..\data\04_analysis_results\visual_features\07_AI_AND_REAL_TIKTOK_VIDEOS_stratified_with_skin_smoothness.csv


,skin_selected_sigma,skin_texture_highpass_var,skin_texture_dog_var,skin_smoothness_highpass_index,skin_smoothness_dog_index,detected_skin_face_frames
0,2.0,103.369597,26.970541,0.009916,0.035782,7
1,2.0,93.408905,26.843148,0.013034,0.041864,10
2,2.0,434.135851,82.712008,0.002738,0.013829,7
3,2.0,332.158933,82.050183,0.003988,0.016213,7
4,2.0,212.726677,28.030090,0.012891,0.076436,7
5,2.0,531.873085,72.020666,0.001958,0.014109,8
6,2.0,82.495046,26.510662,0.012725,0.036418,7
7,2.0,251.032673,45.498608,0.004490,0.022468,8
8,2.0,97.416890,25.937634,0.010804,0.038079,6
9,2.0,50.779866,16.599253,0.019919,0.057392,8
